## ஊடகமாக்கல்கள்

நமது முந்தைய எடுத்துக்காட்டில், நாங்கள் நீளம் `vocab_size` ஆகிய உயர் பரிமாண உரையாடல் பெட்டிகளின் மீது இயங்கினோம், மேலும் குறைந்த பரிமாண நிலை குறியீட்டு வெகுஜனதாள் பிரதிநிதித்துவ வெகுஜனதாள்களுக்கு நேர்மறையாக மாற்றவும் செய்தோம். இந்த ஒன்போட் பிரதிநிதித்துவம் நினைவகம்-திறமைமிக்கதல்ல, கூடுதலாக, ஒவ்வொரு சொலும் மற்றுமொரு சிலபடியில்லாமல் சுயமாக கருதப்படுகிறது, அதாவது ஒன்போட் குறியீட்டமைக்கப்பட்ட வெகுஜனதாள்கள் சொற்களுக்கு இடையேயான எந்தவொரு அர்த்தப்பூர்வ ஒத்திசைவையும் வெளிப்படுத்துவதில்லை.

இந்த அலகில், நாங்கள் தொடர்ந்தும் **செய்தி AG** தரவுத்தொகுதியைப் பார்்கப்போம். தொடங்க, தரவை ஏற்றி முந்தைய குறியீட்டு புத்தகத்திலிருந்து சில வரையறைகளை பெற்றுக்கொள்வோம்.


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## எம்பெட்டிங் என்பது என்ன?

**எம்பெட்டிங்** என்ற கருத்து குறைந்த பரிமாணம் கொண்ட அடர்த்தியான வெக்டர்களாக சொற்களை பிரதிநிதித்துவப்படுத்துகிறது, இது ஒரு வார்த்தையின் பொருள் பூர்வமான அர்த்தத்தை எப்படியோ பிரதிபலிக்கிறது. நாம னுாடி, அர்த்தமுள்ள வார்த்தை எம்பெட்டிங்களை எப்படி உருவாக்குவது என்பதைக் குறித்து விவாதிப்போம், ஆனால் தற்போது எம்பெட்டிங்களை ஒரு வார்த்தை வெக்டரின் பரிமாணத்தைக் குறைக்கக் கூடிய முறையாகவே நினைத்துக்கொள்ளலாம்.

ஆகவே, எம்பெட்டிங் அடுக்கு ஒரு வார்த்தையை உள்ளீடாகப் பெற்று குறிப்பிடப்பட்ட `embedding_size` அளவிலான வெளியீட்டு வெக்டரை உருவாக்கும். ஒருகாரம் பார்க்க, இது `Linear` அடுக்குடன் மிகவும் ஒத்ததாக இருக்கும், ஆனால் ஒருகாரம் ஒன்று-ஹாட் குறியாக்கப்பட்ட வெக்டரை எடுத்துக்கொள்ளவில்லை; அது ஒரு வார்த்தை எண்ணை உள்ளீடாகக் கொள்ளக் கூடியது.

எம்பெட்டிங் அடுப்பை நம் நரம்பியல் வலையில் முதல் அடுப்பு ஆக பயன்படுத்துவதன் மூலம், நாம bag-of-words-ல் இருந்து **எம்பெட்டிங் பாக்** என்ற மாதிரிக்கு மாற்றமடைய முடியும், இதில் நம் உரையிலுள்ள ஒவ்வொரு வார்த்தையையும் அதற்கான எம்பெட்டிங்காக மாற்றி, அதன் பின்னர் அந்த எம்பெட்டிங்களின் அனைத்திற்குமான ஒரு கூட்டு செயல்பாட்டைப் பார்்க்கிறோம், அதாவது `sum`, `average` அல்லது `max` பார்க்க.

![ஐந்து தொடர் வார்த்தைகளுக்கான எம்பெட்டிங் வகைப்படுத்தியொருவை காட்டும் படமானது.](../../../../../translated_images/ta/embedding-classifier-example.b77f021a7ee67eee.webp)

நம் வகைப்படுத்தி நரம்பியல் வலைப்பின்னல் எம்பெட்டிங் அடுப்பு, அதன் பிறகு கூட்டு அடுப்பு, மற்றும் அதன் மேல் நிலை வரிசை வகைப்படுத்தியொருவுடன் தொடங்கும்:


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### மாறும் வரிசை அளவை கையாளுதல்

இந்த வடிவமைப்பின் பேரில, நமது நெட்வொர்க்கிற்கு மினிபேட்ச்களை ஒரு குறிப்பிட்ட முறையில் உருவாக்க வேண்டும். கடந்த அலகில், bag-of-words பயன்படுத்தும்போது, ஒரு மினிபேட்சில் அனைத்து BoW டென்சர்களும் நமது உரை வரிசையின் உண்மையான நீளத்திற்கு படம் பிடிக்காமல் சமமான அளவுக்கு இருந்தன `vocab_size`. வார்த்தை உணர்வுகளுக்கு மாறும் போது, ஒவ்வொரு உரை மாதிரியிலும் மாறும் எண்ணிக்கையிலான வார்த்தைகள் கிடைக்கும், மற்றும் அந்த மாதிரிகளை மினிபேட்ச்களில் இணைக்கும் போது, சில padding பயன்படுத்த வேண்டியிருக்கும்.

இதை datasource-க்கு `collate_fn` செயல்பாட்டை வழங்கும் அதே நுட்பம் மூலம் செய்யலாம்:


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### எம்பெட்டிங் வகைப்படுத்துநரை பயிற்று செய்தல்

இப்போது நாங்கள் சரியான டேட்டாலோடரை வரையறுத்துள்ளோம், முன் அலகில் வரையறுக்கப்பட்ட பயிற்சி செயல்பாட்டைப் பயன்படுத்தி மாதிரியை பயிற்று செய்யலாம்:


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **குறிப்பு**: நேரத்தைக் கருதி, இங்கே 25k பதிவுகள் மட்டுமே பயிற்சி பெறுகிறோம் (ஒரு முழு காலம் குறைவாக), ஆனால் நீங்கள் பயிற்சி தொடரலாம், பல காலங்களுக்கு பயிற்சி செய்ய ஒரு செயல்பாட்டை எழுதலாம், மற்றும் அதிக துல்லியத்தை அடைய கற்றல் விகிதப் பொதுவியலைப் பரிசோதிக்கலாம். நீங்கள் சுமார் 90% துல்லியத்திற்குச் செல்ல முடியும்.  


### EmbeddingBag நிலை மற்றும் மாறிலி நீளம் வரிசை பிரதிநிதித்துவம்

முந்தைய கட்டமைப்பில், நாம் அனைத்து வரிசைகளையும் ஒன்றே நீளத்தில் பத்தியிட வேண்டும் என்று minibatch ஒன்றில் பொருத்தி வேலை செய்துவந்தோம். இது மாறிலி நீள வரிசைகளை பிரதிநிதித்துவம் செய்ய சிறந்த வழி அல்ல - மற்றொரு அணுகுமுறை **offset** வெக்டரைப் பயன்படுத்துவது, இது ஒரு பெரிய வெக்டரில் சேமிக்கப்பட்ட அனைத்து வரிசைகளின் நிலைகளை கையாளும்.

![offset வரிசை பிரதிநிதித்துவத்தை காட்டும் படம்](../../../../../translated_images/ta/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **நோட்டு**: மேல் படத்தில், நாம் எழுத்துக்களின் வரிசையை காட்டுகிறோம், ஆனால் நமது எடுத்துக்காட்டில் வார்த்தைகளின் வரிசைகளுடன் பணியாற்றுகிறோம். இருப்பினும், offset வெக்டரை கொண்டு வரிசைகளை பிரதிநிதித்துவம் செய்வது பற்றிய பொது கொள்கை அதேபோல் உள்ளது.

offset பிரதிநிதித்துவத்துடன் பணியாற்ற [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html) நிலையை பயன்படுத்துகிறோம். இது `Embedding` போலவே உள்ளது, ஆனால் உள்ளடக்க வெக்டர் மற்றும் offset வெக்டரை உள்ளீடாக எடுத்துக்கொள்கிறது, மேலும் இது சராசரி நிலையை உட்படுத்துகிறது, அது `mean`, `sum` அல்லது `max` ஆக இருக்கலாம்.

இங்கே `EmbeddingBag` பயன்படுத்தும் திருத்திய வலைப்பின்னல் உள்ளது:


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

பயிற்சிக்கான தரவுத்தொகுப்பைப் தயாரிக்க, இடைவெளி வெக்டர் தயாரிக்கும் மாற்றும் செயல்பாட்டை வழங்க வேண்டும்:


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

கவனிக்கவும், முந்தைய அனைத்து உதாரணங்களிலும் மாறாக, எங்கள் நெட்வொர்க் இப்போது இரண்டு அளவுருக்களைக் கொள்கிறது: தரவு வெக்டர் மற்றும் மாற்று வெக்டர், அவை வெவ்வேறு அளவுகளோடு உள்ளன. அதேபோல், எங்கள் தரவு ஏற்றியாளரும் 2வற்றின் பதிலாக 3 மதிப்புகளை வழங்குகிறது: உரையும் மாற்று வெக்டர் இரண்டும் அம்சங்களாக வழங்கப்படுகின்றன. எனவே, அதற்கான கவனத்தை எடுக்க எங்கள் பயிற்சி செயலியை சிறிது அளவுக்கு சரிசெய்ய வேண்டும்:


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## அர்த்த வார்த்தைச் சேர்க்கைகள்: Word2Vec

நமது முந்தைய எடுத்துக்காட்டில், மாடல் சேர்க்கை அடுக்கு வார்த்தைகளை வேக்டர் பிரதிநிதித்துவமாக வரைபடம் செய்ய கற்றுக் கொண்டது, எனினும், இந்த பிரதிநிதித்துவத்திற்கு அதிக அர்த்தபூர்வமான பொருள் இல்லை. குறிப்பிட்ட ஆகிய வேக்டர் தூரத்தில் (எ.கா. யூக்லிடியன் தூரம்) ஒத்த வார்த்தைகள் அல்லது பொருத்தமான வார்த்தைகள் ஒரே பரப்பில் அமையும் போன்ற வேக்டர் பிரதிநிதித்துவத்தை கற்றுக்கொள்ளுவது சிறந்தது.

அதற்கு, நம்முடைய சேர்க்கை மாடலை பெரிய எழுத்து தொகுப்பில் குறிப்பிட்ட முறையில் முன்பு பயிற்சி செய்ய வேண்டும். அர்த்த வார்த்தைச் சேர்க்கைகளை பயிற்றுவிப்பதற்கான முதல் முறைகளில் ஒன்றானது [Word2Vec](https://en.wikipedia.org/wiki/Word2vec). இது வார்த்தைகளின் பகிர்ந்த பிரதிநிதித்துவத்தை உருவாக்க இரண்டு முக்கிய கட்டமைப்புகளை அடிப்படையாகக் கொண்டது:

 - **தொடர்ச்சியான வார்த்தை பைல் (Continuous bag-of-words)** (CBoW) — இந்த கட்டமைப்பில், மாடலை சூழல் வாக்கியத்திலிருந்து ஒரு வார்த்தையை முன்னறிவிக்க பயிற்றுவிக்கிறோம். ngram $(W_{-2},W_{-1},W_0,W_1,W_2)$ கொடுக்கப்பட்டு, மாடலின் நோக்கம் $(W_{-2},W_{-1},W_1,W_2)$ இலிருந்து $W_0$ ஐ முன்னறிவிப்பது.
 - **தொடர்ச்சியான skip-gram** என்பது CBoWக்கு எதிர்பார்ப்பு. மாடல் தற்போதைய வார்த்தையை முன்னறிவிக்க சுற்றியுள்ள சூழல் வார்த்தை ஜன்னலைப் பயன்படுத்துகிறது.

CBoW விரைவானது, ஆனால் skip-gram மெதுவானது, ஆனால் அரிதான வார்த்தைகளை பிரதிநிதித்துவப்படுத்துவதில் சிறந்தது.

![இரண்டுமே CBoW மற்றும் Skip-Gram ஆல் வார்த்தைகளை வேக்டர்களாக மாற்றும் விதத்தை காண்பிக்கும் படம்.](../../../../../translated_images/ta/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

Google News தரவுத்தொகுப்பில் முன்புதயாரிக்கப்பட்ட word2vec சேர்க்கையை முயற்சிக்க, **gensim** நூலகத்தை பயன்படுத்தலாம். கீழே 'neural' என்ற வார்த்தைக்கு மிக ஒத்தான வார்த்தைகள் காணப்படுகின்றன.

> **குறிப்பு:** நீங்கள் முதலில் வார்த்தை வேக்டர்களை உருவாக்கும் போது, அவற்றை பதிவிறக்க சில நேரம் எடுக்கலாம்!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


வார்த்தையிலிருந்து வெக்டர் எம்பெடிங்க்களை கணக்கிட முடியும், அவை வகைப்படுத்தும் மாதிரியை பயிற்சி அளிக்க பயன்படுத்தப்படலாம் (வெக்டரின் முதல் 20 கூறுகளையே தெளிவுக்காக காட்டுகிறோம்):


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

அர்த்த சார்ந்த உள்ளடக்கங்களின் சிறப்பு என்னவென்றால், நீங்கள் வெக்டர் குறியீட்டைப் மாற்றி அர்த்தங்களை மாற்ற முடியும். உதாரணமாக, நாம் ஒரு சொல்லை கண்டுபிடிக்க கேட்டு அதற்கான வெக்டர் பிரதிநிதித்துவம் *king* மற்றும் *woman* என்பவைகளுக்கு όσο இலகுவாக இருக்க வேண்டும் மற்றும் *man* என்ற சொல்லிலிருந்து όσο தொலைவில் இருக்க வேண்டும் என்று கேட்கலாம்:


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

CBoW மற்றும் Skip-Grams இரண்டும் "முன்னறிவிப்புச்" சொருக்களாகும், அவை உள்ளூர் சூழலை மட்டுமே கருத்தில் கொள்கின்றன. Word2Vec உலகளாவிய சூழலின் நன்மையைப் பயன்படுத்தாது.

**FastText**, Word2Vec இல் அடிப்படையாக கொண்டு ஒவ்வொரு வார்த்தைக்கும் மற்றும் ஒவ்வொரு வார்த்தையின் உள்ளே காணப்படும் எழுத்து n-grams க்கான வெக்டார் சார்புகளை கற்றுக்கொள்கிறது. பிரதிநிதித்துவங்களின் மதிப்புகள் ஒவ்வொரு பயிற்சி படியிலும் ஒரு வெக்டாராக சராசரி செய்யப்படுகின்றன. இது முன்-பயிற்சிக்கு அதிக கணக்கீட்டுகளை சேர்ப்பினும், வார்த்தை சார்புகள் துணை வார்த்தை தகவலை குறிக்க உதவுகின்றன.

மற்றொரு முறை, **GloVe**, இணைநிலைத் தொலைகாட்சி எண்ணிக்கை என்ற கருத்தை பயன்படுத்தி, இணைநிலைத் தொலைகாட்சி எண்ணிக்கையைக் குறிக்குறிய நியூரல் முறைகளைப் பயன்படுத்தி ஆற்றல் நிறைந்த மற்றும் நேரடியல்லாத வார்த்தை வெக்டார்களாக பிரிக்கிறது.

gensim பல்வேறு வார்த்தை சார்பு மாதிரிகளை ஆதரிக்கும் என்பதால், FastText மற்றும் GloVe க்கு சார்புகளை மாற்றி உதாரணத்துடன் விளையாடலாம்.


## PyTorch இல் முன் பயிற்சி செய்யப்பட்ட உள்ளீடுகளைப் பயன்படுத்துதல்

எங்களுடைய embedding படுக்கையில் முன்னதாக Word2Vec போன்ற அர்த்தவிழித்த உள்ளீடுகளுடன் மாதிரியை முன் நிரப்பக்கூடியவாறு மேலே உள்ள உதாரணத்தை மாற்றலாம். முன் பயிற்சி செய்யப்பட்ட உள்ளீடு மற்றும் எங்கள் உரை தொகுப்பின் சொல் அகராதிகள் பொதுவாக பொருந்தாமல் இருக்கக்கூடும் என்பதைக் கருத்தில் கொள்ள வேண்டும், ஆகையால் காணாமல் இருக்கும் சொற்களுக்கு எடை மதிப்புகளை ரேண்டம் மதிப்புகளுடன் ஆரம்பிப்போம்:


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


இப்போது நாம் எங்கள் மாடலை பயிற்றுவிப்போம். மாடலை பயிற்றுவிக்க விடும் நேரம் முன்னைய உதாரணத்தில் இருந்ததைவிட குறிப்பிடத்தக்க அளவில் அதிகமானது என்பதை கவனியுங்கள், இது பெரிய உள்தரவு அடுக்கின் அளவுக்கு காரணமாகும், ஆகவே பராமரிப்புகளின் எண்ணிக்கை மிகவும் அதிகமாகும். இதன் காரணமாக, அதிகமான எடுத்துக்காட்டுகளில் எங்கள் மாடலை பயிற்றுவிக்க நேரிடலாம், அப்போ தான் நாம் மேற்பாகம் தவிர்க்க முடியும்.


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

எங்கள் வழக்கில், வெவ்வேறு சொல் தொகுப்புகளின் காரணமாக, துல்லியத்தில் மிகப்பெரிய உயர்வு காணவில்லை. 
வெவ்வேறு சொல் தொகுப்புகளின் பிரச்சனையை கடக்க, கீழ்க்கண்ட தீர்வுகளுள் ஒன்றைப் பயன்படுத்தலாம்:
* எங்கள் சொல் தொகுப்பில் word2vec மாதிரியை மீண்டும் பயிற்சி செய்யவும்
* முன்னதாக பயிற்சி பெற்ற word2vec மாதிரியில் இருந்து சொல் தொகுப்புடன் எங்கள் தரவுத்தொகுப்பை ஏற்றவும். தரவுத்தொகுப்பை ஏற்றும்போது பயன்படுத்தப்படும் சொல் தொகுப்பை குறிப்பிடலாம்.

இரண்டாவது அணுகுமுறை எளிமையாக தெரிகிறது, குறிப்பாக PyTorch `torchtext` கட்டமைப்பு உள்ளக உறவுச்சொல் ஆதரவுடன் உள்ளது என்பதால். உதாரணமாக, கீழ்காணும் விதமாக GloVe அடிப்படையிலான சொல் தொகுப்பை உருவாக்கலாம்:


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


Loaded vocabulary has the following basic operations:
* `vocab.stoi` dictionary allows us to convert word into its dictionary index
* `vocab.itos` does the opposite - converts number into word
* `vocab.vectors` is the array of embedding vectors, so to get the embedding of a word `s` we need to use `vocab.vectors[vocab.stoi[s]]`

Here is the example of manipulating embeddings to demonstrate the equation **kind-man+woman = queen** (I had to tweak the coefficient a bit to make it work):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

அந்த எంబெட்டிங்க்களைப் பயன்படுத்தி வகைப்பாட்டாளரை பயிற்றுவிக்க, முதலில் நமது தரவுத்தொகுப்பை GloVe vocabulary-ஐப் பயன்படுத்தி குறியாக்கம் செய்ய வேண்டும்:


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

மேலே நாம் பார்த்ததுபோல், அனைத்து வெக்டர் எம்பெட்டிங்குகளும் `vocab.vectors` מטרிக்தில் சேமிக்கப்படுகின்றன. எம்பெட்டிங் லேயரின் எடைகளை எளிதாக அச்சிடுவதற்கான எடைகளை எளிய நகலெடுக்கும்படி அதைக் பயன்படுத்துவது மிகவும் எளிதாக்குகிறது:


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

இப்போது நமது மாதிரியை பயிற்சி செய்து, நாம் சிறந்த முடிவுகளை பெறுகிறோமா என்று பார்ப்போம்:


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

நாங்கள் முக்கியமான துல்லியமாக்கல் அதிகரிப்பைக் காணாமல் இருப்பதற்கான காரணங்களில் ஒன்று, எங்கள் தரவுத்தொகுப்பில் உள்ள சில சொற்கள் முன்-பயிற்சி பெற்ற GloVe சொல் தொகுப்பில் இல்லாததால் அவை அடிப்படையாகத் தவிர்க்கப்படுகிறதுதான். இந்தச் சிக்கலைத் தாண்ட, நாங்கள் எங்கள் சொந்த உள்ளடக்கங்களை எங்கள் தரவுத்தொகுப்பில் பயிற்சி அளிக்கலாம்.


## சூழலியல் இறுக்குமுகங்கள்

Word2Vec போன்ற பாரம்பரிய முன்பயிற்சி செய்யப்பட்ட இறுக்குமுக பிரதிநிதித்துவங்களில் ஒரு முக்கியமான வரம்பு என்னவென்றால் சொல் பொருள் தெளிவு செய்வதில் சிக்கல் காணப்படுவது ஆகும். முன்பயிற்சி செய்யப்பட்ட இறுக்குமுகங்கள் ஒரு சொல் சூழலில் பொருளை சில அளவிற்கு பிடிக்கக்கூடியவையாக இருந்தாலும், அந்த வார்த்தையின் அனைத்து சாத்தியமான பொருள்களும் ஒரே இறுக்குமுகத்தில் குறியிடப்பட்டுள்ளன. இது கீழ்த் தரப்படும் மாதிரிகளுக்கு பிரச்சனைகளை ஏற்படுத்தும், ஏனெனில் 'play' என்ற சொல்லைப் போலவே பல சொற்கள் பயன்படுத்தப்படும் சூழலில் வெவ்வேறு அர்த்தங்களை கொண்டிருக்கும்.

எடுத்துக்காட்டாக, கீழ்க்காணும் இரண்டு வாக்கியங்களில் 'play' என்ற сөзத்தின் அர்த்தங்கள் முற்றிலும் வேறுபட்டவை:
- நான் ஒரு **play** கலைநாடகத்திற்கு சென்றேன்.
- ஜான் தனது நண்பர்களுடன் **play** செய்ய விரும்புவர்.

மேல் குறிப்பிட்ட முன்பயிற்சி செய்யப்பட்ட இறுக்குமுகங்கள் 'play' என்ற சொல்லின் இரு பொருள்களையும் ஒரே இறுக்குமுகத்தில் பிரதிநிதித்துவம் செய்கின்றன. இந்த வரம்பை கடக்க, நாம் ஒரு பெரிய உரை தொகுப்பில் பயிற்சி பெற்ற **மொழி மாதிரி** அடிப்படையில் இறுக்குமுகங்களை உருவாக்க வேண்டியிருக்கும், இதன் மூலம் சொற்கள் வெவ்வேறு சூழல்களில் எவ்வாறு சேர்க்கப்படுகின்றன என்பதைக் *கற்றுக்கொள்ள* முடியும். சூழலியல் இறுக்குமுகங்களை விவாதிப்பது இந்த பயிற்சிக்குறிப்புக்கு வெளியில் உள்ளது, ஆனால் நாங்கள் அடுத்த அலகில் மொழி மாதிரிகள் பற்றி பேசும் போது மீண்டும் இதை எடுத்துக்கொள்ளப்போகின்றோம்.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
